In [ ]:
# If you're running this the first time in a fresh env you may need:
# !pip install numpy pandas scikit-learn xgboost

from pathlib import Path
import numpy as np
import pandas as pd

from katebatic.models.medgan.medgan import Medgan

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


In [ ]:
# Choose any dataset that follows katabatic/sample_data/<dataset> layout
DATASET = "adult"         # e.g. "adult", "magic", "car", ...

# MedGAN expects either "binary" or "count" features.
# - If your features are already one-hot/binary → "binary"
# - If features are non-negative integers/floats → "count"
DATA_TYPE = "count"       # change to "binary" if your X are 0/1

# Where to save MedGAN run artifacts (relative, repo-safe)
RUN_DIR = Path("runs") / f"{DATASET}_medgan"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def find_data_folder(dataset: str) -> Path:
    """
    Robustly locate `katabatic/sample_data/<dataset>`, regardless of where the
    notebook is opened from within the repo.
    """
    here = Path.cwd().resolve()
    candidates = []
    for root in [here, *here.parents]:
        candidates.append(root / "katabatic" / "sample_data" / dataset)
        candidates.append(root / "sample_data" / dataset)
    data = next((p for p in candidates if p.exists()), None)
    assert data is not None, (
        f"Couldn't find 'sample_data/{dataset}' under any of: {candidates}"
    )
    return data

DATA = find_data_folder(DATASET)
print("Using data folder:", DATA.as_posix())


In [ ]:
# Expect x_train.csv, y_train.csv, x_test.csv, y_test.csv
X_tr = pd.read_csv(DATA / "x_train.csv")
y_tr = pd.read_csv(DATA / "y_train.csv").squeeze("columns")
X_te = pd.read_csv(DATA / "x_test.csv")
y_te = pd.read_csv(DATA / "y_test.csv").squeeze("columns")

print("Shapes:",
      "\n  X_tr:", X_tr.shape,
      "\n  y_tr:", y_tr.shape,
      "\n  X_te:", X_te.shape,
      "\n  y_te:", y_te.shape)

# Ensure numeric & non-negative for "count" data type
if DATA_TYPE == "count":
    X_tr = X_tr.fillna(0)
    X_te = X_te.fillna(0)
    # Make sure non-negative
    X_tr = np.maximum(0, X_tr).astype(np.float32)
    X_te = np.maximum(0, X_te).astype(np.float32)

if DATA_TYPE == "binary":
    # Convert to 0/1 if needed (threshold > 0)
    X_tr = (X_tr > 0).astype(np.int8)
    X_te = (X_te > 0).astype(np.int8)


In [ ]:
# Initialize adapter (repo_root is '.' because medgan.py lives inside the repo)
medgan = Medgan(
    repo_root=".",              # Katabatic repo root
    run_dir=RUN_DIR,            # where to keep checkpoints
    data_type=DATA_TYPE,
    n_pretrain_epoch=10,        # keep short for demo; adjust in practice
    n_epoch=20,
    batch_size=128,
    save_max_keep=5,
)

# MedGAN is unsupervised → train on all available features (train + test combined, or just train)
X_all = np.vstack([np.asarray(X_tr), np.asarray(X_te)])
print("Fitting MedGAN on X_all:", X_all.shape)
_ = medgan.fit(X_all)


In [ ]:
# Sample roughly the training set size (you can change this)
N_SYN = len(X_tr)

X_syn = medgan.sample(n=N_SYN)
print("Synthetic shape:", X_syn.shape)

# Save an artifact for downstream reuse (optional)
np.save(RUN_DIR / f"{DATASET}.synthetic.npy", X_syn, allow_pickle=False)


In [ ]:
# ----- Label the synthetic rows -----
# Basic, reproducible baseline: resample y_train to match synthetic size
rng = np.random.RandomState(0)
y_syn = y_tr.sample(n=len(X_syn), replace=True, random_state=rng).to_numpy()

# ----- Define simple evaluation models -----
models = {
    "LogReg": LogisticRegression(max_iter=200, n_jobs=None, class_weight="balanced", solver="lbfgs"),
    "MLP":    MLPClassifier(hidden_layer_sizes=(128,), max_iter=50, random_state=0),
    "RF":     RandomForestClassifier(n_estimators=200, random_state=0),
}

# XGBoost (optional)
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, random_state=0, eval_metric="logloss",
        tree_method="hist", n_jobs=0
    )
except Exception:
    pass

# ----- Evaluate -----
def eval_binary(y_true, proba, pred):
    acc = accuracy_score(y_true, pred)
    f1  = f1_score(y_true, pred, average="binary")
    try:
        auc = roc_auc_score(y_true, proba[:, 1] if proba.ndim == 2 else proba)
    except Exception:
        auc = np.nan
    return acc, f1, auc

results = []
for name, clf in models.items():
    # Fit on synthetic
    clf.fit(X_syn, y_syn)

    # Predict on real test
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X_te)
    else:
        # Some models don’t expose predict_proba; use decision_function as a proxy if available
        if hasattr(clf, "decision_function"):
            s = clf.decision_function(X_te)
            # Convert scores to a "probability-like" 2-col array for metrics compatibility
            # (min-max scale to [0,1]; not a true probability, but OK for AUC computation)
            s = (s - s.min()) / (s.max() - s.min() + 1e-8)
            proba = np.vstack([1 - s, s]).T
        else:
            # fall back to uniform 0.5 proba if absolutely nothing is available
            proba = np.full((len(X_te), 2), 0.5)

    pred = clf.predict(X_te)

    acc, f1, auc = eval_binary(y_te, proba, pred)
    results.append((name, acc, f1, auc))
    print(f"{name:10s}  Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")

# Nicely formatted summary
pd.DataFrame(results, columns=["Model", "Accuracy", "F1", "AUC"]).sort_values("AUC", ascending=False)
